# YOLO + Pose + ArUco + 추적 + 경로 예측 (v8 · PIAI 담당 부분)

## 이 노트북의 범위
- YOLO pose: 작업자 검출 + 스켈레톤
- YOLO custom: 지게차 / box_1 / box_2 검출
- ArUco: 작업자 신원 매칭
- BoT-SORT: 작업자 추적
- 선형 외삽: 작업자·지게차 미래 위치 예측
- 충돌 판정: 예측 원 겹침 여부

## 다음 조원 담당 (주석처리됨)
- 픽셀 → 절대좌표 변환 (`pixel_to_absolute`)
- 5프레임마다 딥러닝 모델로 JSON 전송 (`send_to_deeplearning`)

## 객체별 처리
| 객체 | 박스 | 밑변 중심점 | 경로 예측 |
|---|---|---|---|
| worker | ✓ (pose가 그림) | ✓ | ✓ |
| forklift | ✓ | ✓ | ✓ |
| box_1 | ✓ | ✗ | ✗ |
| box_2 | ✓ | ✗ | ✗ |


In [1]:
import cv2
import os
import time
import math
import json
import numpy as np
from collections import deque
from ultralytics import YOLO
from dotenv import load_dotenv

load_dotenv()
best_model_path = os.getenv("best_model_path")

In [2]:
# ========================================
# 경로 예측 파라미터 (시간 기반)
# ========================================

HISTORY_SECONDS = 0.5
PREDICT_SECONDS = 2.0
MIN_HISTORY_SECONDS = 0.3

COLLISION_RADIUS = 250

YOLO_IMGSZ = 480
WINDOW_W = 1600
WINDOW_H = 900

WINDOW_NAME = "Pose + Forklift + ArUco + Prediction"

# ========================================
# 딥러닝 전송 설정 (다음 조원 담당 · 현재 미사용)
# ========================================

# CAMERA_ID = 1              # 이 노트북이 담당하는 카메라 번호 (1, 2, 3 중)
# SEND_EVERY_N_FRAMES = 5    # 몇 프레임마다 딥러닝 모델에 전송할지

print(f"설정 완료: {PREDICT_SECONDS}초 후 예측")


설정 완료: 2.0초 후 예측


In [3]:
# ========================================
# 모델 로드
# ========================================

pose_model = YOLO("yolo11n-pose.pt")
forklift_model = YOLO(best_model_path)

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_50)
aruco_params = cv2.aruco.DetectorParameters()
aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

worker_map = {
    0: "Worker1",
    22: "Worker2",
    24: "Worker3",
    27: "Worker4",
    38: "Worker5",
}

track_to_worker = {}
print("모델 로드 완료")

모델 로드 완료


In [4]:
# ========================================
# 좌표 추출 함수(객체 발밑으로 예측하기 위해)
# ========================================

def get_bottom_center(x1, y1, x2, y2):
    """박스 밑변 중심 좌표 (픽셀).
    작업자: 양발 위치 / 지게차: 바닥 접지면."""
    cx = (x1 + x2) // 2
    cy = y2
    return cx, cy


# ========================================
# 절대좌표 변환 (다음 조원 담당 · 현재 미구현)
# ========================================

# def pixel_to_absolute(px, py):
#     """픽셀 좌표 → 맵 절대좌표 (m) 변환.
#     ArUco 기반 Homography 행렬 사용 예정."""
#     # TODO: ArUco 기반 Homography 계산 후 변환
#     # H = cv2.findHomography(pixel_corners, world_corners)
#     # world_pt = cv2.perspectiveTransform([[[px, py]]], H)
#     # return world_pt[0][0][0], world_pt[0][0][1]
#     return 0.0, 0.0


# ========================================
# 예측 관련 함수
# ========================================

def update_history(history_dict, obj_id, cx, cy, max_len):
    if obj_id not in history_dict:
        history_dict[obj_id] = deque(maxlen=max_len)
    elif history_dict[obj_id].maxlen != max_len:
        old_data = list(history_dict[obj_id])
        history_dict[obj_id] = deque(old_data, maxlen=max_len)
    history_dict[obj_id].append((cx, cy))


def predict_future_position(history, predict_frames, min_history):
    if len(history) < min_history:
        return None

    old_x, old_y = history[0]
    cur_x, cur_y = history[-1]
    n_frames = len(history) - 1

    if n_frames == 0:
        return None

    vx = (cur_x - old_x) / n_frames
    vy = (cur_y - old_y) / n_frames

    future_x = int(cur_x + vx * predict_frames)
    future_y = int(cur_y + vy * predict_frames)

    return (future_x, future_y)


def is_collision_risk(pos1, pos2, radius=COLLISION_RADIUS):
    if pos1 is None or pos2 is None:
        return False
    distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
    return distance < radius


def draw_prediction(frame, current_pos, future_pos, color, label=""):
    if future_pos is None:
        return

    cx, cy = current_pos
    fx, fy = future_pos

    num_dots = 10
    for i in range(num_dots):
        t1 = i / num_dots
        t2 = (i + 0.5) / num_dots
        p1 = (int(cx + (fx - cx) * t1), int(cy + (fy - cy) * t1))
        p2 = (int(cx + (fx - cx) * t2), int(cy + (fy - cy) * t2))
        cv2.line(frame, p1, p2, color, 2)

    cv2.circle(frame, (fx, fy), 15, color, 2)

    if label:
        cv2.putText(frame, label, (fx + 18, fy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)


# ========================================
# 딥러닝 모델 전송 (다음 조원 담당 · 현재 미구현)
# ========================================

# def send_to_deeplearning(payload):
#     """
#     TODO: 딥러닝 모델로 JSON payload 전송.
#     구현 옵션: HTTP POST / WebSocket / ZeroMQ / 파일 queue
#     팀원과 프로토콜 확정 후 구현.
#     """
#     # requests.post("http://딥러닝서버/api/coords", json=payload)
#     # 또는 websocket.send(json.dumps(payload))
#     print(f"[SEND] {json.dumps(payload, ensure_ascii=False)}")


print("함수 정의 완료")


함수 정의 완료


In [5]:
# ========================================
# 메인 루프
# ========================================

worker_history = {}
forklift_history = {}

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, WINDOW_W, WINDOW_H)

cap = cv2.VideoCapture(1)

# FPS 측정용
fps_counter = 0
fps_start_time = time.time()
current_fps = 30.0

# 프레임 카운터 (딥러닝 전송용 · 현재 미사용 / 다음 조원)
# frame_count = 0

# 색상 상수
NORMAL_COLOR = (0, 255, 0)
DANGER_COLOR = (0, 0, 255)

# 클래스 ID 매핑
CLS_FORKLIFT = 0
CLS_BOX_1 = 1
CLS_BOX_2 = 2

print("루프 시작 - ESC 누르면 종료")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # frame_count += 1   # 딥러닝 전송용 · 현재 미사용 (다음 조원)

    # FPS 측정
    fps_counter += 1
    elapsed = time.time() - fps_start_time
    if elapsed >= 1.0:
        current_fps = fps_counter / elapsed
        fps_counter = 0
        fps_start_time = time.time()

    # FPS 기반 동적 파라미터
    history_len = max(int(HISTORY_SECONDS * current_fps), 5)
    predict_frames = int(PREDICT_SECONDS * current_fps)
    min_history = max(int(MIN_HISTORY_SECONDS * current_fps), 3)

    # YOLO 추론
    pose_results = pose_model.track(
        frame, conf=0.25, persist=True, verbose=False, classes=[0],
        imgsz=YOLO_IMGSZ
    )
    forklift_results = forklift_model(
        frame, conf=0.5, verbose=False,
        imgsz=YOLO_IMGSZ
    )

    # ============================================
    # 작업자 박스 + 궤적 (밑변 중심 기준)
    # ============================================
    person_boxes = []
    if (pose_results[0].boxes is not None
            and pose_results[0].boxes.id is not None):
        xyxy = pose_results[0].boxes.xyxy.cpu().numpy()
        ids_t = pose_results[0].boxes.id.cpu().numpy().astype(int)
        for box, tid in zip(xyxy, ids_t):
            x1, y1, x2, y2 = box.astype(int)
            person_boxes.append((tid, x1, y1, x2, y2))

            # 밑변 중심 (양발 위치)
            bx, by = get_bottom_center(x1, y1, x2, y2)
            update_history(worker_history, tid, bx, by, history_len)

    # ============================================
    # 지게차(추적) vs 박스(시각화만) 분리 처리
    # ============================================
    forklift_boxes = []   # cls_id==0 지게차만 · 추적/예측 대상
    static_boxes = []     # cls_id==1(box_1), 2(box_2) · 시각화만
    
    for i, box in enumerate(forklift_results[0].boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0]) if hasattr(box, 'cls') else 0
        
        if cls_id == CLS_FORKLIFT:
            forklift_boxes.append((i, x1, y1, x2, y2))
            # 밑변 중심 (가로 중심, 바닥 접지면)
            bx, by = get_bottom_center(x1, y1, x2, y2)
            update_history(forklift_history, i, bx, by, history_len)
        else:
            # box_1, box_2: 박스만 그리기 · 밑변 점/예측 없음
            static_boxes.append((i, x1, y1, x2, y2, cls_id))

    # ============================================
    # ArUco 검출 + 매칭
    # ============================================
    corners, ids, _ = aruco_detector.detectMarkers(frame)
    if ids is not None:
        for marker_corners, marker_id in zip(corners, ids.flatten()):
            marker_id = int(marker_id)
            if marker_id not in worker_map:
                continue
            pts = marker_corners[0]
            cx = int(pts[:, 0].mean())
            cy = int(pts[:, 1].mean())
            for tid, x1, y1, x2, y2 in person_boxes:
                if x1 <= cx <= x2 and y1 <= cy <= y2:
                    track_to_worker[tid] = worker_map[marker_id]
                    break

    # ============================================
    # 시각화
    # ============================================
    annotated_frame = pose_results[0].plot()

    # 지게차 박스 + 밑변 중심 점
    for i, x1, y1, x2, y2 in forklift_boxes:
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(annotated_frame, "forklift", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        # 밑변 중심 점 (지게차만)
        bx, by = get_bottom_center(x1, y1, x2, y2)
        cv2.circle(annotated_frame, (bx, by), 4, (0, 255, 255), -1)

    # 박스 (box_1, box_2): 박스만 · 밑변 점 없음
    for i, x1, y1, x2, y2, cls_id in static_boxes:
        label = f"box_{cls_id}"
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 128, 0), 2)
        cv2.putText(annotated_frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 128, 0), 2)

    # 작업자 이름 + 밑변 중심 점
    for tid, x1, y1, x2, y2 in person_boxes:
        name = track_to_worker.get(tid)
        if name is None:
            display_text = "Unknown"
            color = (128, 128, 128)
        else:
            display_text = name
            color = (0, 255, 255)
        cv2.putText(annotated_frame, display_text, (x1, y1 - 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        # 밑변 중심 점
        bx, by = get_bottom_center(x1, y1, x2, y2)
        cv2.circle(annotated_frame, (bx, by), 4, (0, 255, 255), -1)

    if ids is not None:
        cv2.aruco.drawDetectedMarkers(annotated_frame, corners, ids)

    # ============================================
    # 경로 예측 + 충돌 판정 + 시각화 (작업자 + 지게차만)
    # ============================================
    worker_predictions = {}
    forklift_predictions = {}
    pred_label = f"+{PREDICT_SECONDS:.1f}s"

    for tid, x1, y1, x2, y2 in person_boxes:
        if tid not in worker_history:
            continue
        future = predict_future_position(
            worker_history[tid], predict_frames, min_history
        )
        worker_predictions[tid] = future

    for i, x1, y1, x2, y2 in forklift_boxes:
        if i not in forklift_history:
            continue
        future = predict_future_position(
            forklift_history[i], predict_frames, min_history
        )
        forklift_predictions[i] = future

    # 충돌 판정 (작업자 vs 지게차만)
    collision_worker_ids = set()
    collision_forklift_ids = set()
    for tid, w_future in worker_predictions.items():
        for idx, f_future in forklift_predictions.items():
            if is_collision_risk(w_future, f_future):
                collision_worker_ids.add(tid)
                collision_forklift_ids.add(idx)

    # 작업자 예측 원
    for tid, x1, y1, x2, y2 in person_boxes:
        if tid not in worker_history:
            continue
        current = worker_history[tid][-1]
        future = worker_predictions.get(tid)
        color = DANGER_COLOR if tid in collision_worker_ids else NORMAL_COLOR
        draw_prediction(annotated_frame, current, future,
                        color=color, label=pred_label)

    # 지게차 예측 원
    for i, x1, y1, x2, y2 in forklift_boxes:
        if i not in forklift_history:
            continue
        current = forklift_history[i][-1]
        future = forklift_predictions.get(i)
        color = DANGER_COLOR if i in collision_forklift_ids else NORMAL_COLOR
        draw_prediction(annotated_frame, current, future,
                        color=color, label=pred_label)

    # ============================================
    # 딥러닝 모델로 JSON 전송 (다음 조원 담당 · 현재 미구현)
    # ============================================
    # if frame_count % SEND_EVERY_N_FRAMES == 0:
    #     objects_payload = []
    #
    #     # 작업자
    #     for tid, x1, y1, x2, y2 in person_boxes:
    #         bx, by = get_bottom_center(x1, y1, x2, y2)
    #         abs_x, abs_y = pixel_to_absolute(bx, by)
    #         name = track_to_worker.get(tid, f"Unknown_{tid}")
    #         objects_payload.append({
    #             "type": "worker",
    #             "id": name,
    #             "center_abs": [abs_x, abs_y],
    #         })
    #
    #     # 지게차
    #     for i, x1, y1, x2, y2 in forklift_boxes:
    #         bx, by = get_bottom_center(x1, y1, x2, y2)
    #         abs_x, abs_y = pixel_to_absolute(bx, by)
    #         objects_payload.append({
    #             "type": "forklift",
    #             "id": f"forklift_{i:02d}",
    #             "center_abs": [abs_x, abs_y],
    #         })
    #
    #     # 박스 (box_1, box_2)
    #     for i, x1, y1, x2, y2, cls_id in static_boxes:
    #         bx, by = get_bottom_center(x1, y1, x2, y2)
    #         abs_x, abs_y = pixel_to_absolute(bx, by)
    #         type_name = f"box_{cls_id}"
    #         objects_payload.append({
    #             "type": type_name,
    #             "id": f"{type_name}_{i:02d}",
    #             "center_abs": [abs_x, abs_y],
    #         })
    #
    #     # 페이로드 최종 조립
    #     payload = {
    #         "timestamp": round(time.time(), 3),
    #         "frame_id": frame_count,
    #         "camera_id": CAMERA_ID,
    #         "objects": objects_payload,
    #     }
    #
    #     send_to_deeplearning(payload)

    cv2.imshow(WINDOW_NAME, annotated_frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

print(f"종료. 최종 FPS: {current_fps:.1f}")


루프 시작 - ESC 누르면 종료
종료. 최종 FPS: 3.9
